# B02 · Data pull v2: U.S. products (with package size), prices, and category taxonomy

This notebook rebuilds the three input files for the analysis notebook directly from the source datasets:

| Output (in `DATA_DIR`) | What it is | Approx. size |
|---|---|---|
| `off_prices_us.csv` | Latest Open Prices observations recorded at U.S. stores (all 48 original columns) | ~25 MB |
| `off_products_us.parquet` + `off_products_us.csv` | One row per product: U.S. products **plus any product that has a U.S. price**, with clean text fields, package size, labels, stores, food groups, and 9 nutrients | ~1M rows, ~40 columns |
| `categories.json`, `food_groups.json` | Official Open Food Facts taxonomies (categories and food groups) | ~60 MB, <1 MB |

**New in v2 (for the final deliverable)**
- `size_g_ml`: package size normalized to grams or milliliters. It comes from `product_quantity` / `product_quantity_unit`, or is parsed from the `quantity` text (e.g. "16 oz", "1.5 L") when those are empty. This makes **price per 100 g** possible.
- `is_organic` from the official `labels_tags` (en:organic, en:usda-organic, en:eu-organic), replacing the product-name keyword search.
- `ingredients_analysis_tags` (vegan, vegetarian, palm-oil-free) for later segment checks.
- Section 1b downloads the source files once, which avoids Hugging Face's `HTTP 429` rate limit.

**What changed from the first pull
- **Columns:**
  - Multilingual fields (`product_name`, `ingredients_text`) are flattened to plain text in SQL. The old CSV stored Python-style lists like `[{'lang': 'main', 'text': ...}]`.
  - Lists (`categories_tags`, `labels_tags`, …) are written as `|`-separated text in the CSV and as real lists in the Parquet file.
- **New columns:**
  - `product_quantity` / `product_quantity_unit` / `quantity` / `serving_size`, needed for **price per 100 g**.
  - `food_groups_tags`, the official food group, computed by Open Food Facts.
  - `labels_tags` (organic, non-GMO, …), `stores_tags`, `ingredients_n`, `additives_n`, `nutriscore_score`, `completeness`, `unique_scans_n`, `obsolete`.
  - Saturated fat, carbohydrates, fiber, and sodium.
- **Rows:**
  - U.S. products (`countries_tags` contains `en:united-states`), **plus** any product whose barcode has a U.S. price even if it isn't tagged as sold in the U.S. In our first join, about 4,700 priced barcodes had no product row for this reason.
  - Duplicate barcodes keep the most recently edited row.
  - Non-food (beauty) products are excluded.

**How to run:** Colab → *Runtime → Run all*. Step 1b downloads the 7.9 GB product file to Colab's temporary disk (a few minutes), then DuckDB reads only the columns it needs, so memory use stays low. If Colab restarts, the download runs again.

In [ ]:
!pip -q install duckdb huggingface_hub pyarrow

## 1. Settings

In [ ]:
import json, os, sys, time
from pathlib import Path

import duckdb
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    default_dir = "/content/drive/MyDrive/BA780 Team 2/Codebase/input_data"
else:
    default_dir = "data"
DATA_DIR = Path(os.environ.get("OFF_DATA_DIR", default_dir))
DATA_DIR.mkdir(parents=True, exist_ok=True)

FOOD_PARQUET = os.environ.get("OFF_FOOD_PARQUET", "hf://datasets/openfoodfacts/product-database/food.parquet")
PRICES_REPO = "openfoodfacts/open-prices"
PRICES_PARQUET = os.environ.get("OFF_PRICES_PARQUET")        # found automatically below if not set
COUNTRY_TAG = "en:united-states"
PRICE_COUNTRY_CODE = "US"
TAXONOMY_URL = "https://static.openfoodfacts.org/data/taxonomies/{name}.json"
INCLUDE_INGREDIENTS_TEXT = False   # True adds the full ingredient text (~+300 MB CSV); ingredients_n is enough for counts

# Nutrients to flatten from the `nutriments` list: output column -> Open Food Facts nutrient name
NUTRIENT_FIELDS = {
    "energy-kcal_100g": "energy-kcal", "fat_100g": "fat", "saturated-fat_100g": "saturated-fat",
    "carbohydrates_100g": "carbohydrates", "sugars_100g": "sugars", "fiber_100g": "fiber",
    "proteins_100g": "proteins", "salt_100g": "salt", "sodium_100g": "sodium",
}
print("Food parquet:", FOOD_PARQUET)
print("Output folder:", DATA_DIR)

## 1b. Download the source files to local disk
Reading the 7.9 GB parquet straight from Hugging Face sends thousands of small requests, which triggers
Hugging Face's rate limit (`HTTP 429 Too Many Requests`). Downloading each file once avoids that, and the
queries below run much faster on a local file. In Colab the files go to `/content` (temporary local disk), not to Drive.

*Optional:* a free Hugging Face token gives higher limits. Create one at huggingface.co/settings/tokens and
add it as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

# Use the HF_TOKEN Colab secret if one exists (optional)
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

LOCAL_DIR = Path("/content/hf_files" if IN_COLAB else "hf_files")

def download(repo, filename):
    # Resumable download with automatic retries; skipped if the file is already there
    t = time.time()
    path = hf_hub_download(repo, filename, repo_type="dataset", local_dir=LOCAL_DIR)
    print(f"{filename}: {Path(path).stat().st_size / 1e9:.2f} GB ready in {time.time() - t:.0f}s")
    return path

if FOOD_PARQUET.startswith("hf://"):
    FOOD_PARQUET = download("openfoodfacts/product-database", "food.parquet")

if PRICES_PARQUET is None:
    files = [f for f in HfApi().list_repo_files(PRICES_REPO, repo_type="dataset") if f.endswith(".parquet")]
    print("Parquet files in the Open Prices repo:", files)
    PRICES_PARQUET = download(PRICES_REPO, files[0])

## 2. Connect DuckDB and check the source schema

In [ ]:
con = duckdb.connect()
if any(str(s).startswith(("hf://", "http")) for s in (FOOD_PARQUET, PRICES_PARQUET or "hf://")):
    con.execute("INSTALL httpfs; LOAD httpfs;")   # needed to read Hugging Face files over HTTP
con.execute("SET preserve_insertion_order = false;")

# Main-language text from a list of {lang, text} structs (falls back to English, then the first entry)
con.execute('''
    CREATE OR REPLACE MACRO main_text(x) AS coalesce(
        list_filter(x, e -> e.lang = 'main')[1].text,
        list_filter(x, e -> e.lang = 'en')[1].text,
        x[1].text);
''')
# Per-100 g value of one nutrient from the `nutriments` list of structs
con.execute('''
    CREATE OR REPLACE MACRO nutrient_100g(nutriments, nutrient_name) AS
        list_filter(nutriments, x -> x.name = nutrient_name)[1]."100g";
''')

schema = con.execute(f"DESCRIBE SELECT * FROM '{FOOD_PARQUET}'").df()
col_type = dict(zip(schema["column_name"], schema["column_type"]))
print(f"Source has {len(col_type)} columns.")
wanted = ["code", "product_name", "brands", "brands_tags", "categories_tags", "food_groups_tags", "labels_tags",
          "stores_tags", "countries_tags", "quantity", "product_quantity", "product_quantity_unit", "serving_size",
          "serving_quantity", "nutriscore_grade", "nutriscore_score", "nova_group", "environmental_score_grade",
          "environmental_score_score", "ingredients_text", "ingredients_n", "additives_n", "completeness",
          "unique_scans_n", "popularity_key", "obsolete", "created_t", "last_modified_t", "ingredients_analysis_tags",
          "nutriments"]
missing = [c for c in wanted if c not in col_type]
print("Missing (will be skipped):", missing or "none")
pd.Series({c: col_type[c] for c in wanted if c in col_type}, name="type").to_frame()

## 3. Prices: latest U.S. observations from Open Prices
The Open Prices dump is refreshed daily, so re-running this cell picks up new prices.

In [ ]:
t = time.time()
prices_us = con.execute(f'''
    SELECT * FROM '{PRICES_PARQUET}'
    WHERE location_osm_address_country_code = '{PRICE_COUNTRY_CODE}'
''').arrow().read_all().to_pandas()
prices_us.to_csv(DATA_DIR / "off_prices_us.csv", index=False)
priced_codes = (prices_us.loc[prices_us["type"].eq("PRODUCT"), "product_code"].dropna().astype(str)
                .str.split(".").str[0].str.lstrip("0").unique())
con.register("priced_codes", pd.DataFrame({"code_norm": priced_codes}))
print(f"U.S. prices: {len(prices_us):,} rows × {prices_us.shape[1]} columns | "
      f"{len(priced_codes):,} distinct priced barcodes | {time.time() - t:.0f}s")

## 4. Products: build the query
Each output column is chosen by its type in the source, so the query still works if a column is a list in one release and plain text in another.

In [ ]:
def is_struct_list(c):
    return c in col_type and "STRUCT" in col_type[c].upper() and col_type[c].endswith("[]")

def is_list(c):
    return c in col_type and col_type[c].endswith("[]")

select = []
for c in wanted:
    if c not in col_type or c == "nutriments" or (c == "ingredients_text" and not INCLUDE_INGREDIENTS_TEXT):
        continue
    if is_struct_list(c):
        select.append(f'main_text("{c}") AS "{c}"')                       # [{lang, text}] -> text
    elif c in ("product_quantity", "serving_quantity", "nutriscore_score", "environmental_score_score",
               "ingredients_n", "additives_n", "completeness", "unique_scans_n", "nova_group"):
        select.append(f'TRY_CAST("{c}" AS DOUBLE) AS "{c}"')              # numbers stored as text in some releases
    else:
        select.append(f'"{c}"')
select.append(f"list_contains(countries_tags, '{COUNTRY_TAG}') AS in_us_catalog")
select.append("ltrim(code, '0') IN (SELECT code_norm FROM priced_codes) AS has_us_price")
select += [f'TRY_CAST(nutrient_100g(nutriments, \'{n}\') AS DOUBLE) AS "{alias}"' for alias, n in NUTRIENT_FIELDS.items()]

# Package size: use product_quantity (+ unit) when present, otherwise parse the free-text `quantity`
# ("16 oz", "1.5 L", "500 g"). Everything is converted to grams or milliliters; implausible sizes become NULL.
SIZE_RE = r"([0-9]+(?:[.,][0-9]+)?)\s*(fl\.?\s?oz|oz|lbs?|kg|gal(?:lon)?s?|qt|pt|g|ml|cl|l)\b"
PACK_RE = r"([0-9]+)\s*[x×]\s*[0-9]"             # multipacks such as "6 x 1.5 oz" -> multiply by 6
TO_G_ML = {"g": 1, "ml": 1, "kg": 1000, "l": 1000, "cl": 10, "oz": 28.3495, "lb": 453.592, "lbs": 453.592,
           "floz": 29.5735, "gal": 3785.41, "gals": 3785.41, "gallon": 3785.41, "gallons": 3785.41,
           "qt": 946.353, "pt": 473.176}
factor_sql = "CASE size_unit " + " ".join(f"WHEN '{u}' THEN {f}" for u, f in TO_G_ML.items()) + " END"
organic_sql = ("list_has_any(labels_tags, ['en:organic', 'en:usda-organic', 'en:eu-organic'])"
               if is_list("labels_tags") else "NULL::BOOLEAN")

products_query = f'''
    WITH base AS (
        SELECT {", ".join(select)}
        FROM '{FOOD_PARQUET}'
        WHERE list_contains(countries_tags, '{COUNTRY_TAG}')
           OR ltrim(code, '0') IN (SELECT code_norm FROM priced_codes)
        QUALIFY row_number() OVER (PARTITION BY code ORDER BY last_modified_t DESC NULLS LAST) = 1
    ),
    sized AS (
        SELECT *,
            CASE WHEN product_quantity > 0 THEN product_quantity
                 ELSE TRY_CAST(replace(regexp_extract(lower(quantity), '{SIZE_RE}', 1), ',', '.') AS DOUBLE)
                      * coalesce(TRY_CAST(regexp_extract(lower(quantity), '{PACK_RE}', 1) AS DOUBLE), 1) END AS size_value,
            replace(replace(CASE WHEN product_quantity > 0 THEN coalesce(lower(product_quantity_unit), 'g')
                 ELSE regexp_extract(lower(quantity), '{SIZE_RE}', 2) END, '.', ''), ' ', '') AS size_unit,
            CASE WHEN product_quantity > 0 THEN 'product_quantity'
                 WHEN regexp_matches(lower(quantity), '{SIZE_RE}') THEN 'quantity text' END AS size_source,
            {organic_sql} AS is_organic
        FROM base
    )
    SELECT * EXCLUDE (size_value),
           CASE WHEN size_value * {factor_sql} BETWEEN 1 AND 25000 THEN size_value * {factor_sql} END AS size_g_ml
    FROM sized
'''
print(products_query[:2500], "…")

## 5. Products: run and save (Parquet keeps lists and types; CSV is for the analysis notebook)

In [ ]:
t = time.time()
out_parquet = DATA_DIR / "off_products_us.parquet"
con.execute(f"COPY ({products_query}) TO '{out_parquet.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f"Parquet written in {time.time() - t:.0f}s")

# CSV version: lists become '|'-separated text so they survive the round trip
csv_cols = [f'array_to_string("{c}", \'|\') AS "{c}"' if is_list(c) and not is_struct_list(c) else f'"{c}"'
            for c in con.execute(f"DESCRIBE SELECT * FROM '{out_parquet.as_posix()}'").df()["column_name"]]
con.execute(f'''COPY (SELECT {", ".join(csv_cols)} FROM '{out_parquet.as_posix()}')
               TO '{(DATA_DIR / "off_products_us.csv").as_posix()}' (HEADER, DELIMITER ',')''')
for f in ("off_products_us.parquet", "off_products_us.csv", "off_prices_us.csv"):
    print(f"{f:28s} {(DATA_DIR / f).stat().st_size / 1e6:8.1f} MB")

## 6. Sanity checks

In [ ]:
p = f"'{out_parquet.as_posix()}'"
out_cols = set(con.execute(f"DESCRIBE SELECT * FROM {p}").df()["column_name"])
counts = {"products": "count(*)", "distinct_codes": "count(DISTINCT code)",
          "us_tagged": "sum(in_us_catalog::INT)", "added_because_priced": "sum((NOT in_us_catalog)::INT)",
          "with_us_price": "sum(has_us_price::INT)"}
shares = {"nutriscore": ("nutriscore_grade", "nutriscore_grade IN ('a','b','c','d','e')"),
          "nova": ("nova_group", "nova_group BETWEEN 1 AND 4"),
          "green_score": ("environmental_score_grade", "environmental_score_grade IN ('a-plus','a','b','c','d','e','f')"),
          "package_size": ("size_g_ml", "size_g_ml > 0"),
          "organic_label": ("is_organic", "is_organic"),
          "categories": ("categories_tags", "len(categories_tags) > 0"),
          "food_group": ("food_groups_tags", "len(food_groups_tags) > 0")}
exprs = [f"{e} AS {k}" for k, e in counts.items()]
exprs += [f"avg(coalesce({cond}, false)::INT) AS pct_{k}" for k, (col, cond) in shares.items() if col in out_cols]
checks = con.execute(f"SELECT {', '.join(exprs)} FROM {p}").df().T.rename(columns={0: "value"})
checks["value"] = [f"{v:.1%}" if k.startswith("pct_") else f"{v:,.0f}" for k, v in checks["value"].items()]
print("Among priced products:")
print(con.execute(f'''
    SELECT count(*) AS priced_products,
           avg(coalesce(size_g_ml > 0, false)::INT) AS pct_package_size,
           avg(coalesce(nutriscore_grade IN ('a','b','c','d','e'), false)::INT) AS pct_nutriscore
    FROM {p} WHERE has_us_price''').df().round(3).to_string(index=False))
checks

In [ ]:
# Where package sizes came from, and a quick look at the unit mix (should be mostly g and ml)
print(con.execute(f"SELECT size_source, count(*) AS n FROM {p} GROUP BY 1 ORDER BY n DESC").df().to_string(index=False))
con.execute(f'''SELECT size_unit, count(*) AS n, median(size_g_ml) AS median_size_g_ml
               FROM {p} GROUP BY 1 ORDER BY n DESC LIMIT 8''').df()

## 7. Official category taxonomies
`categories.json` holds about 14,700 categories, each with its English name, `parents`, `children`, and properties such as `food_groups`. `food_groups.json` holds the three-level food-group hierarchy. The analysis notebook explains how they are used.

In [ ]:
import urllib.request
for name in ("categories", "food_groups"):
    dest = DATA_DIR / f"{name}.json"
    try:
        urllib.request.urlretrieve(TAXONOMY_URL.format(name=name), dest)
        print(f"{name}.json: {len(json.load(open(dest, encoding='utf-8'))):,} entries, {dest.stat().st_size / 1e6:.1f} MB")
    except Exception as e:
        print(f"Could not download {name}.json ({e}). Download it manually from {TAXONOMY_URL.format(name=name)}")

**Next:** open the analysis notebook. It reads `off_products_us.csv`, `off_prices_us.csv`, `categories.json`, and `food_groups.json` from the same folder.

*Data: Open Food Facts and Open Prices, Open Database License (ODbL) 1.0.*